In [1]:
from __future__ import annotations

import json
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold

from features import FEATURE_COLUMNS, CATEGORICAL_FEATURE_NAMES, FeatureBuilder


In [15]:
ROOT = Path.cwd()
DATA = ROOT
REPORTS = ROOT / "reports"
MODEL_PARAMS = dict(
    objective="regression",
    metric="mae",
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=25,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.3,
    random_state=42,
    verbosity=-1,
)
HOLDOUT_START = "2025-09-14"

In [4]:
def build_model() -> lgb.LGBMRegressor:
    return lgb.LGBMRegressor(**MODEL_PARAMS)

In [5]:
def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mape": float(mean_absolute_percentage_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


In [11]:
def time_based_holdout(df: pd.DataFrame) -> dict:
    train_part = df[df["date"] < HOLDOUT_START].reset_index(drop=True)
    holdout_part = df[df["date"] >= HOLDOUT_START].reset_index(drop=True)

    builder = FeatureBuilder().fit(train_part)
    train_feat = builder.transform(train_part)
    holdout_feat = builder.transform(holdout_part)

    model = build_model()
    model.fit(
        train_feat[FEATURE_COLUMNS],
        np.log1p(train_feat["posted_rate"]),
        eval_set=[(holdout_feat[FEATURE_COLUMNS], np.log1p(holdout_feat["posted_rate"]))],
        eval_metric="mae",
        categorical_feature=CATEGORICAL_FEATURE_NAMES,
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
    )

    pred_log = model.predict(holdout_feat[FEATURE_COLUMNS], num_iteration=model.best_iteration_)
    pred = np.expm1(pred_log)
    metrics = evaluate(holdout_feat["posted_rate"].values, pred)
    metrics["n_train"] = int(len(train_part))
    metrics["n_holdout"] = int(len(holdout_part))
    metrics["holdout_date_range"] = [holdout_part["date"].min(), holdout_part["date"].max()]
    metrics["best_iteration"] = int(model.best_iteration_)

    importances = sorted(
        zip(FEATURE_COLUMNS, model.feature_importances_.tolist()), key=lambda x: -x[1]
    )
    metrics["top_features"] = importances[:10]
    return metrics

In [12]:
def random_kfold_cv(df: pd.DataFrame, n_splits: int = 5) -> dict:
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_metrics = []
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(df)):
        train_part = df.iloc[train_idx].reset_index(drop=True)
        test_part = df.iloc[test_idx].reset_index(drop=True)

        builder = FeatureBuilder().fit(train_part)
        train_feat = builder.transform(train_part)
        test_feat = builder.transform(test_part)

        model = build_model()
        model.set_params(n_estimators=600)  # fixed budget, no early stopping needed for CV sanity check
        model.fit(
            train_feat[FEATURE_COLUMNS],
            np.log1p(train_feat["posted_rate"]),
            categorical_feature=CATEGORICAL_FEATURE_NAMES,
        )
        pred = np.expm1(model.predict(test_feat[FEATURE_COLUMNS]))
        fold_metrics.append(evaluate(test_feat["posted_rate"].values, pred))

    avg = {k: float(np.mean([m[k] for m in fold_metrics])) for k in fold_metrics[0]}
    return {"per_fold": fold_metrics, "average": avg}


In [13]:
def train_final_model(df: pd.DataFrame, best_iteration: int) -> tuple[lgb.LGBMRegressor, FeatureBuilder]:
    builder = FeatureBuilder().fit(df)
    feat = builder.transform(df)
    model = build_model()
    model.set_params(n_estimators=max(best_iteration, 200))
    model.fit(
        feat[FEATURE_COLUMNS],
        np.log1p(feat["posted_rate"]),
        categorical_feature=CATEGORICAL_FEATURE_NAMES,
    )
    return model, builder


In [16]:
def main() -> None:
    REPORTS.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(DATA / "train_test.csv")

    print("=== Time-based holdout (primary validation) ===")
    holdout_metrics = time_based_holdout(df)
    print(json.dumps({k: v for k, v in holdout_metrics.items() if k != "top_features"}, indent=2))
    print("Top features:")
    for name, score in holdout_metrics["top_features"]:
        print(f"  {name}: {score}")

    print("\n=== 5-fold random CV (secondary sanity check) ===")
    cv_metrics = random_kfold_cv(df)
    print(json.dumps(cv_metrics["average"], indent=2))

    with open(REPORTS / "validation_metrics.json", "w") as f:
        json.dump({"time_based_holdout": holdout_metrics, "random_cv": cv_metrics}, f, indent=2)

    print("\n=== Refitting final model on all labeled data ===")
    final_model, final_builder = train_final_model(df, holdout_metrics["best_iteration"])

    import joblib

    joblib.dump({"model": final_model, "builder": final_builder}, REPORTS / "final_model.joblib")
    print(f"Saved final model to {REPORTS / 'final_model.joblib'}")


if __name__ == "__main__":
    main()


=== Time-based holdout (primary validation) ===
{
  "mae": 126.14937050050952,
  "rmse": 603.4411292346583,
  "mape": 0.0567542099488911,
  "r2": 0.8398276194037972,
  "n_train": 40542,
  "n_holdout": 7458,
  "holdout_date_range": [
    "2025-09-14",
    "2025-10-31"
  ],
  "best_iteration": 179
}
Top features:
  distance: 1269
  quote_signal: 996
  delivery_code: 935
  pickup_code: 929
  weight_clean: 926
  equipment_code: 859
  lane_code: 857
  market_index_clean: 796
  day_of_year: 576
  haversine_miles: 494

=== 5-fold random CV (secondary sanity check) ===
{
  "mae": 119.22245370358966,
  "rmse": 601.1868239907265,
  "mape": 0.054227553270544224,
  "r2": 0.8361822630896757
}

=== Refitting final model on all labeled data ===
Saved final model to /content/reports/final_model.joblib
